# 제조 음성 ASR 심사 파이프라인

하나의 노트북에서 세 가지 데이터 모드를 사용합니다.

- `PUBLIC_PROXY`: 공개 Zeroth 한국어 음성으로 데이터 준비부터 모델 비교, 자동 프록시 선택, LoRA, 양자화, 고정 Test, 보고서·백데이터까지 전체 동작을 검증합니다.
- `SYNTHETIC_MANUFACTURING`: 저장소의 한국어 TTS 제조 문장 600개로 실제 제조 데이터를 넣기 전 동일한 코드 경로를 검증합니다.
- `PRIVATE_MANUFACTURING`: 나중에 승인된 제조 녹음과 검수 전사로 같은 코드를 다시 실행합니다. 이 모드의 모델·양자화 선택은 반드시 사람이 수행합니다.

> 공개·합성 결과는 코드와 산출물의 정상 동작 증거입니다. 제조 현장 성능, 배포 적합성 또는 심사 최종 결론의 증거로 사용하면 안 됩니다.

양자화, BM25 정보검색, 벡터 최근접 이웃 검색, 지식 증류는 실행 모드와 별도로 각각 켜고 끌 수 있습니다. 지식 증류는 추가 GPU 시간과 모델 다운로드가 필요하므로 기본값은 꺼짐입니다.

**보안:** 카메라·마이크·패스키를 사용하지 않습니다. 실제 음성은 GitHub에 올리지 않고 승인된 비공개 Drive 경로만 사용합니다.

## 0. 실행 모드

In [ ]:
# "PUBLIC_PROXY", "SYNTHETIC_MANUFACTURING", "PRIVATE_MANUFACTURING"
# UTC 기준 실행 시각을 생성하는 표준 날짜·시간 클래스를 불러옵니다.
from datetime import UTC, datetime

# 현재 실행을 합성 제조 TTS 데이터 모드로 선택합니다. 실제 데이터 사용 시 값을 바꿉니다.
DATA_MODE = "SYNTHETIC_MANUFACTURING"

# 선택된 Whisper의 LoRA 학습 단계를 실행하도록 켭니다.
RUN_LORA = True
# LoRA가 자원 한도로 실패해도 양자화·최종 Test 증거 생성을 계속하도록 허용합니다.
LORA_FAILURE_IS_NON_BLOCKING = True
# 선택 모델의 정밀도별 양자화 비교 단계를 실행하도록 켭니다.
RUN_QUANTIZATION = True
# Validation에서 안전한 보정 조합을 탐색하고 합격 설정만 선택하도록 켭니다.
RUN_CORRECTION_SWEEP = True
# 보정 탐색 후보에 BM25 기반 제조 용어 정보검색을 포함합니다.
ENABLE_INFORMATION_RETRIEVAL = True
# 보정 탐색 후보에 벡터 최근접 이웃 검색을 포함합니다.
ENABLE_NEAREST_NEIGHBOR = True
# Teacher·Student 지식 증류는 비용이 커 기본 비활성화하며 필요할 때 True로 바꿉니다.
RUN_DISTILLATION = False  # 추가 GPU 비용이 큰 선택 실험

# False이면 같은 설계 버전의 완료 결과를 재사용하고, True이면 새 시간 ID로 전체 실험을 분리합니다.
FORCE_NEW_EXPERIMENT = False
# 새 실험 모드일 때만 세션 ID를 생성하며, 같은 노트북 실행 안에서는 값이 바뀌지 않습니다.
EXPERIMENT_SESSION_ID = (
    # 연도부터 초까지의 UTC 시각을 파일명에 안전한 형식으로 변환합니다.
    datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ")
    # 사용자가 새 실험 생성을 요청한 경우에만 시간 접미사를 활성화합니다.
    if FORCE_NEW_EXPERIMENT
    # 기존 결과 재사용 모드에서는 설계 ID를 그대로 유지합니다.
    else None
)

# Colab에서 복제할 ASR 프로젝트 GitHub 저장소 주소를 지정합니다.
GITHUB_REPO_URL = "https://github.com/Pronesis9758/aias-specialist-asr.git"
# 벤치마크·양자화 기능이 있는 작업 브랜치를 선택합니다. PR 병합 후 main으로 바꿉니다.
GITHUB_BRANCH = "codex/whisper-benchmark-quantization"  # PR 병합 후 main
# GitHub 코드를 복제할 Colab 임시 경로입니다. 런타임 종료·초기화 시 삭제됩니다.
PROJECT_DIR = "/content/AIAS"
# 모델·결과·보고서를 보존할 내 Google Drive 경로입니다. 런타임 종료 후에도 유지됩니다.
DRIVE_ROOT = "/content/drive/MyDrive/AI_Specialist_ASR_Project"

# 데이터 출처별 설정·모델 후보·양자화 명세·실험 ID를 하나의 표로 정의합니다.
MODE_SETTINGS = {
    # 공개 Zeroth 한국어 음성으로 전체 파이프라인만 검증하는 프록시 모드를 정의합니다.
    "PUBLIC_PROXY": {
        # 공개 Zeroth 데이터·평가·Drive 산출물 경로가 담긴 설정 파일을 연결합니다.
        "config": "configs/public_proxy_assessment.yaml",
        # 공개 프록시에서 비교할 Whisper 후보와 실행 조건 목록을 지정합니다.
        "matrix": "configs/benchmarks/public_proxy_whisper_models.yaml",
        # 공개 프록시 모델에 적용할 float16·int8 양자화 비교 조건을 지정합니다.
        "quantization": "configs/quantization/public_proxy_whisper_quantization.yaml",
        # 공개 프록시 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "public-proxy-whisper-model-benchmark-v1",
        # 공개 프록시 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "public-proxy-whisper-quantization-v1",
    },
    # 제조 용어·난이도·소음 조건을 균형화한 합성 TTS 600개를 사용합니다.
    # 실제 제조 데이터 투입 전 전체 연구 기능을 검증하는 기본 실행 모드입니다.
    "SYNTHETIC_MANUFACTURING": {
        # 합성 제조 데이터 분할·학습·평가 조건이 담긴 설정 파일을 연결합니다.
        "config": "configs/synthetic_manufacturing_sample.yaml",
        # 합성 제조 데이터에서 tiny부터 large-v3까지 정확도·자원 후보 목록을 지정합니다.
        "matrix": "configs/benchmarks/synthetic_manufacturing_whisper_models.yaml",
        # 합성 제조 선택 모델의 float16·int8-float16 비교 조건을 지정합니다.
        "quantization": "configs/quantization/synthetic_manufacturing_whisper_quantization.yaml",
        # 선택 모델의 Validation 예측에서 안전한 보정 조합을 탐색할 명세를 지정합니다.
        "correction_sweep": "configs/correction/synthetic_manufacturing_correction_sweep.yaml",
        # 합성 제조 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "synthetic-manufacturing-whisper-model-benchmark-v6",
        # 합성 제조 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "synthetic-manufacturing-whisper-quantization-v6",
    },
    # 승인된 실제 제조 녹음과 사람 검수 전사를 사용하는 최종 연구 모드를 정의합니다.
    "PRIVATE_MANUFACTURING": {
        # 실제 제조 데이터 경로와 엄격한 거버넌스 조건을 입력할 템플릿을 연결합니다.
        "config": "configs/manufacturing_private_template.yaml",
        # 실제 제조 데이터에서 사람이 검토할 확장 Whisper 후보 목록을 지정합니다.
        "matrix": "configs/benchmarks/manufacturing_whisper_models_template.yaml",
        # 실제 제조 선택 모델에 적용할 양자화 후보와 허용 손실 조건을 지정합니다.
        "quantization": "configs/quantization/manufacturing_whisper_quantization_template.yaml",
        # 실제 제조 모델 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "benchmark_id": "manufacturing-whisper-model-benchmark-v1",
        # 실제 제조 양자화 비교 산출물을 모을 고유 실험 ID를 지정합니다.
        "quantization_id": "manufacturing-whisper-quantization-v1",
    },
}
# 지원 목록에 없는 데이터 모드를 조기에 차단합니다.
if DATA_MODE not in MODE_SETTINGS:
    # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
    raise ValueError(f"지원하지 않는 DATA_MODE: {DATA_MODE}")

# 선택한 데이터 모드의 설정 묶음만 꺼냅니다.
mode = MODE_SETTINGS[DATA_MODE]
# 선택 모드의 데이터·학습·평가 설정 파일 경로를 사용합니다.
CONFIG = mode["config"]
# 선택 모드에서 비교할 Whisper 후보 목록 경로를 사용합니다.
MODEL_MATRIX = mode["matrix"]
# 선택 모델에 적용할 양자화 후보와 평가 조건 파일 경로를 사용합니다.
QUANTIZATION_SPEC = mode["quantization"]
# 현재 모드에 보정 탐색 명세가 있으면 경로를 사용하고 없으면 탐색을 건너뜁니다.
CORRECTION_SWEEP_SPEC = mode.get("correction_sweep")
# 설계 ID에 선택적 시간 접미사를 일관되게 붙이는 재사용 함수를 정의합니다.
def experiment_id(design_id):
    # 새 실험이면 UTC 세션 ID를 붙이고, 재사용 모드이면 원래 설계 ID를 반환합니다.
    return (
        # 설계 의미와 실제 실행 시각을 모두 추적할 수 있는 실험 ID를 만듭니다.
        f"{design_id}-{EXPERIMENT_SESSION_ID}"
        # 공통 세션 ID가 생성된 경우에만 새 결과 폴더를 사용합니다.
        if EXPERIMENT_SESSION_ID
        # 동일 설계 재실행에서는 기존 폴더와 캐시를 재사용합니다.
        else design_id
    )

# 모델 비교 설계 ID에 현재 실행 정책을 적용해 실제 결과 폴더 ID를 만듭니다.
BENCHMARK_ID = experiment_id(mode["benchmark_id"])
# 양자화 설계 ID에도 동일한 세션 ID를 적용해 관련 산출물을 연결합니다.
QUANTIZATION_ID = experiment_id(mode["quantization_id"])
# 실제 제조 음성·정답·승인 문서를 둘 비공개 Drive 폴더를 지정합니다.
PRIVATE_ROOT = f"{DRIVE_ROOT}/data/private/manufacturing"
# 현재 실행이 공개 데이터 기능 검증 모드인지 표시합니다.
IS_PUBLIC_PROXY = DATA_MODE == "PUBLIC_PROXY"
# 현재 실행이 합성 제조 데이터 기능 검증 모드인지 표시합니다.
IS_SYNTHETIC_MANUFACTURING = DATA_MODE == "SYNTHETIC_MANUFACTURING"
# 공개·합성 모드에서는 사람 결정 대신 자동 선택을 허용하도록 표시합니다.
IS_AUTOMATED_PROXY = IS_PUBLIC_PROXY or IS_SYNTHETIC_MANUFACTURING
# 사용자가 확인할 수 있도록 현재 데이터 모드를 출력합니다.
print("Mode:", DATA_MODE)
# 현재 모드가 사용하는 핵심 YAML 설정 경로를 출력합니다.
print("Config:", CONFIG)
# 재사용 여부와 공통 실행 시각을 확인할 수 있도록 실험 식별자를 출력합니다.
print("Experiment session:", EXPERIMENT_SESSION_ID or "reuse-design-id")
# 이후 산출물 경로 확인을 위해 실제 벤치마크 ID를 출력합니다.
print("Benchmark ID:", BENCHMARK_ID)
# 이후 산출물 경로 확인을 위해 실제 양자화 ID를 출력합니다.
print("Quantization ID:", QUANTIZATION_ID)

## 1. GPU와 Google Drive 연결

In [ ]:
# 할당된 GPU 종류와 메모리 상태를 확인합니다.
!nvidia-smi
# pathlib 모듈에서 필요한 기능을 불러옵니다.
from pathlib import Path
# google.colab 모듈에서 필요한 기능을 불러옵니다.
from google.colab import drive

# Colab 런타임에서 Google Drive를 연결할 기준 폴더를 지정합니다.
DRIVE_MOUNT_POINT = Path("/content/drive")
# Drive 연결 성공 여부를 판별할 내 드라이브 폴더 경로를 지정합니다.
DRIVE_MY_DRIVE = DRIVE_MOUNT_POINT / "MyDrive"

# 내 드라이브 폴더가 이미 보이면 추가 승인 없이 기존 연결을 재사용합니다.
if DRIVE_MY_DRIVE.is_dir():
    # 이 런타임은 이미 Drive에 연결됐으므로 추가 승인 없이 재사용한다고 알립니다.
    print("Google Drive already mounted at /content/drive")
# 기존 Drive 연결이 없으므로 새 인증과 마운트를 시도합니다.
else:
    # Google Drive 인증 실패를 사용자가 이해할 수 있는 안내로 변환합니다.
    try:
        # 새 런타임에 모델·산출물을 보존할 Google Drive 접근 승인을 요청합니다.
        drive.mount(str(DRIVE_MOUNT_POINT))
    # Colab의 Drive 마운트 실패 예외를 잡아 브라우저 해결 방법을 안내합니다.
    except ValueError as exc:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise RuntimeError(
            # 인증 팝업 렌더링 실패의 대표 증상을 오류 안내 첫 문장에 포함합니다.
            "Google Drive 연결에 실패했습니다. accounts.google.com 화면이 검게 "
            # 문제가 반복될 때 사용할 정상 브라우저와 대상 노트북을 안내합니다.
            "멈추면 Codex 내장 브라우저 대신 일반 Chrome에서 이 노트북을 "
            # Chrome에서 허용할 권한과 재실행할 단계를 안내합니다.
            "열고 팝업·리디렉션을 허용한 뒤 이 셀을 다시 실행하세요."
        # 사용자 안내 오류에 Colab의 원래 마운트 실패 원인을 연결해 보존합니다.
        ) from exc

# 승인 이후에도 내 드라이브 폴더가 없으면 연결 실패로 처리합니다.
if not DRIVE_MY_DRIVE.is_dir():
    # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
    raise RuntimeError("Google Drive 승인이 완료됐지만 /content/drive/MyDrive를 찾을 수 없습니다.")
# 검증을 통과한 내 드라이브 경로를 출력해 저장 준비 완료를 확인합니다.
print("Google Drive ready:", DRIVE_MY_DRIVE)

## 2. GitHub 코드 동기화

In [ ]:
# os 모듈을 불러옵니다.
import os
# subprocess 모듈을 불러옵니다.
import subprocess

# Colab 임시 디스크에 저장소가 아직 없는지 확인합니다.
if not os.path.exists(PROJECT_DIR):
    # 지정 브랜치의 GitHub 저장소를 Colab 임시 경로에 처음 복제합니다.
    subprocess.run(
        # Git clone 실행 파일·하위 명령·옵션·원본·대상 순서로 명령 목록을 구성합니다.
        [
            # 운영체제에 설치된 Git 실행 파일을 호출합니다.
            "git",
            # 원격 저장소를 Colab 임시 디스크로 복제하는 clone 명령을 선택합니다.
            "clone",
            # 전체 브랜치 중 실행할 특정 브랜치를 지정하는 옵션입니다.
            "--branch",
            # 앞에서 지정한 벤치마크·양자화 작업 브랜치 이름을 전달합니다.
            GITHUB_BRANCH,
            # 선택 브랜치 이력만 받아 다운로드 시간과 디스크 사용량을 줄입니다.
            "--single-branch",
            # 복제 원본인 AIAS GitHub 저장소 주소를 전달합니다.
            GITHUB_REPO_URL,
            # 저장소를 복제하거나 Git 명령을 실행할 Colab 로컬 경로를 전달합니다.
            PROJECT_DIR,
        ],
        # Git·CLI 명령이 실패하면 다음 셀로 진행하지 않고 즉시 예외를 발생시킵니다.
        check=True,
    )
# 저장소가 이미 있으므로 재복제하지 않고 원격 최신 코드와 동기화합니다.
else:
    # 이미 복제된 저장소에서 원격 브랜치의 최신 커밋 정보를 가져옵니다.
    subprocess.run(
        # 프로젝트 폴더의 origin에서 선택 브랜치 최신 이력만 가져오는 Git 명령을 구성합니다.
        ["git", "-C", PROJECT_DIR, "fetch", "origin", GITHUB_BRANCH],
        # Git·CLI 명령이 실패하면 다음 셀로 진행하지 않고 즉시 예외를 발생시킵니다.
        check=True,
    )
    # 기존 저장소의 작업 브랜치를 노트북이 요구하는 브랜치로 전환합니다.
    subprocess.run(
        # 프로젝트 폴더에서 지정 작업 브랜치로 전환하는 Git 명령을 구성합니다.
        ["git", "-C", PROJECT_DIR, "checkout", GITHUB_BRANCH],
        # Git·CLI 명령이 실패하면 다음 셀로 진행하지 않고 즉시 예외를 발생시킵니다.
        check=True,
    )
    # 원격 최신 커밋을 fast-forward 방식으로 로컬 코드에 반영합니다.
    subprocess.run(
        # Git 저장소 위치·merge 방식·원격 브랜치를 순서대로 명령 목록에 담습니다.
        [
            # 운영체제에 설치된 Git 실행 파일을 호출합니다.
            "git",
            # 현재 셸 위치를 바꾸지 않고 지정 저장소에서 Git을 실행하는 옵션입니다.
            "-C",
            # 저장소를 복제하거나 Git 명령을 실행할 Colab 로컬 경로를 전달합니다.
            PROJECT_DIR,
            # 원격 브랜치의 최신 커밋을 현재 체크아웃에 반영합니다.
            "merge",
            # 로컬 변경과 충돌하면 새 merge commit을 만들지 않고 안전하게 실패시킵니다.
            "--ff-only",
            # 동기화할 origin 원격 브랜치 이름을 구성합니다.
            f"origin/{GITHUB_BRANCH}",
        ],
        # Git·CLI 명령이 실패하면 다음 셀로 진행하지 않고 즉시 예외를 발생시킵니다.
        check=True,
    )
# 이후 상대 경로가 저장소를 기준으로 동작하도록 작업 폴더를 바꿉니다.
os.chdir(PROJECT_DIR)
# 재현성을 위해 실제 실행 중인 Git 커밋 해시를 출력합니다.
print("Git commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())

## 3. 의존성 설치와 실행 함수

In [ ]:
# Colab 기본 패키지 중 충돌 가능성이 있는 항목을 제거합니다.
%pip uninstall -y torchao gradio gradio-client
# 프로젝트와 학습용 의존성을 현재 Colab 런타임에 설치합니다.
%pip install -q -e ".[train]" "transformers>=4.46,<5" "peft>=0.14,<0.19"

# importlib.util 모듈에서 필요한 기능을 불러옵니다.
from importlib.util import find_spec
# sys 모듈을 불러옵니다.
import sys

# editable 설치 직후 프로젝트 패키지를 찾을 src 절대 경로를 계산합니다.
PROJECT_SRC = os.path.join(PROJECT_DIR, "src")
# 프로젝트 src 경로가 Python 검색 경로에 없는지 확인합니다.
if PROJECT_SRC not in sys.path:
    # 방금 설치한 저장소의 src를 Python 검색 경로 최우선 순위에 추가합니다.
    sys.path.insert(0, PROJECT_SRC)
# AIAS 패키지를 실제로 import할 수 있는지 확인합니다.
if find_spec("aias_specialist") is None:
    # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
    raise ModuleNotFoundError("aias_specialist import path refresh failed")
# 실제로 import되는 프로젝트 소스 경로를 확인 로그로 출력합니다.
print("aias_specialist import OK:", PROJECT_SRC)


# run_aias 재사용 함수를 정의합니다.
def run_aias(*args):
    # 현재 Python으로 AIAS CLI와 전달받은 세부 명령을 실행할 명령 배열을 만듭니다.
    command = [sys.executable, "-m", "aias_specialist.cli", *args]
    # 실행할 AIAS 명령 전체를 즉시 출력해 장시간 작업의 시작점을 보여줍니다.
    print("\nRunning:", " ".join(command), flush=True)
    # 학습·평가 진행 로그가 지연 없이 Colab에 표시되도록 실행 환경을 만듭니다.
    environment = {**os.environ, "PYTHONUNBUFFERED": "1"}
    # 로그 즉시 출력 환경으로 AIAS 명령을 실행하고 실패 시 파이프라인을 중단합니다.
    subprocess.run(command, check=True, env=environment)

### 3-1. 선택 기능을 실행용 설정에 반영

GitHub 원본 YAML은 수정하지 않고 `/content`에 실행용 사본을 만듭니다. `FORCE_NEW_EXPERIMENT=False`이면 동일 설계 ID의 완료 결과를 재사용하고, `True`이면 한 번 생성한 UTC 세션 ID를 벤치마크·보정·양자화에 함께 붙여 새 실험으로 분리합니다. 따라서 위 Boolean 값만 바꿔 양자화·IR·NN Search·Distillation을 독립적으로 실행할 수 있습니다. `char_ngram` NN은 다운로드 없는 재현용 벡터 검색이며, 실제 임베딩 모델을 쓰려면 YAML의 backend를 `transformers`로 변경합니다.

In [ ]:
# yaml 모듈을 불러옵니다.
import yaml

# 선택 데이터 모드의 원본 YAML을 읽기 위한 Path 객체를 만듭니다.
source_config_path = Path(CONFIG)
# GitHub 원본을 보존하면서 기능 옵션을 적용할 실행용 설정 사본을 읽습니다.
runtime_config = yaml.safe_load(source_config_path.read_text(encoding="utf-8"))
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_config.setdefault("correction", {})
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_config["correction"].setdefault("information_retrieval", {})["enabled"] = (
    # 보정 탐색 전 모델 비교에서는 과도한 IR을 끄고, 탐색을 끈 경우에만 직접 토글을 사용합니다.
    ENABLE_INFORMATION_RETRIEVAL and not RUN_CORRECTION_SWEEP
)
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_config["correction"].setdefault("nearest_neighbor", {})["enabled"] = (
    # 보정 탐색 전 모델 비교에서는 NN을 끄고, 탐색을 끈 경우에만 직접 토글을 사용합니다.
    ENABLE_NEAREST_NEIGHBOR and not RUN_CORRECTION_SWEEP
)
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_config.setdefault("distillation", {})["enabled"] = RUN_DISTILLATION
# 준비도 점검이 현재 세션의 벤치마크 결과 폴더를 찾도록 실행 ID를 반영합니다.
runtime_config.setdefault("assessment", {})["benchmark_id"] = BENCHMARK_ID
# 준비도 점검이 현재 세션의 양자화 결과 폴더를 찾도록 실행 ID를 반영합니다.
runtime_config["assessment"]["quantization_id"] = QUANTIZATION_ID
# 기능 토글이 반영된 임시 설정 YAML의 Colab 경로를 지정합니다.
runtime_config_path = Path("/content/aias_runtime_config.yaml")
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_config_path.write_text(
    # 이 줄의 값을 바로 위에서 설명한 연구 단계의 입력으로 사용합니다.
    yaml.safe_dump(runtime_config, allow_unicode=True, sort_keys=False),
    # 한글이 손상되지 않도록 실행용 YAML을 UTF-8로 저장합니다.
    encoding="utf-8",
)

# 모델 비교가 실행용 설정을 사용하도록 원본 모델 행렬 YAML을 읽습니다.
runtime_matrix = yaml.safe_load(Path(MODEL_MATRIX).read_text(encoding="utf-8"))
# 모델 비교 산출물이 현재 세션 전용 폴더에 저장되도록 벤치마크 ID를 교체합니다.
runtime_matrix["benchmark"]["id"] = BENCHMARK_ID
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_matrix["benchmark"]["base_config"] = str(runtime_config_path)
# 실행용 base config가 연결된 임시 모델 행렬 경로를 지정합니다.
runtime_matrix_path = Path("/content/aias_runtime_model_matrix.yaml")
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_matrix_path.write_text(
    # 이 줄의 값을 바로 위에서 설명한 연구 단계의 입력으로 사용합니다.
    yaml.safe_dump(runtime_matrix, allow_unicode=True, sort_keys=False),
    # 한글이 손상되지 않도록 실행용 YAML을 UTF-8로 저장합니다.
    encoding="utf-8",
)

# 양자화 사용 여부와 실행용 base config를 반영할 원본 명세를 읽습니다.
runtime_quantization = yaml.safe_load(Path(QUANTIZATION_SPEC).read_text(encoding="utf-8"))
# 양자화 산출물에도 벤치마크와 동일한 세션 접미사를 사용하도록 ID를 교체합니다.
runtime_quantization["quantization"]["id"] = QUANTIZATION_ID
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_quantization["quantization"]["enabled"] = RUN_QUANTIZATION
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_quantization["quantization"]["base_config"] = str(runtime_config_path)
# 양자화 토글이 반영된 임시 양자화 명세 경로를 지정합니다.
runtime_quantization_path = Path("/content/aias_runtime_quantization.yaml")
# 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
runtime_quantization_path.write_text(
    # 이 줄의 값을 바로 위에서 설명한 연구 단계의 입력으로 사용합니다.
    yaml.safe_dump(runtime_quantization, allow_unicode=True, sort_keys=False),
    # 한글이 손상되지 않도록 실행용 YAML을 UTF-8로 저장합니다.
    encoding="utf-8",
)

# 현재 실행 모드에 보정 탐색 명세가 있는 경우에만 실행용 사본을 만듭니다.
if CORRECTION_SWEEP_SPEC:
    # 원본 보정 후보와 합격 기준을 보존하면서 실행 ID만 바꿀 YAML을 읽습니다.
    runtime_correction_sweep = yaml.safe_load(
        # 선택 모드에 연결된 보정 탐색 명세를 UTF-8로 불러옵니다.
        Path(CORRECTION_SWEEP_SPEC).read_text(encoding="utf-8")
    )
    # 보정 설계 버전에도 벤치마크와 같은 UTC 세션 접미사를 적용합니다.
    correction_sweep_id = experiment_id(
        # 원본 YAML의 사람이 관리하는 보정 설계 ID를 입력으로 사용합니다.
        runtime_correction_sweep["correction_sweep"]["id"]
    )
    # 보정 로그와 비교표가 현재 세션 전용 폴더에 저장되도록 ID를 교체합니다.
    runtime_correction_sweep["correction_sweep"]["id"] = correction_sweep_id
    # 상대 base_config가 저장소 루트를 기준으로 해석되도록 원본 설정 폴더 아래에 둡니다.
    runtime_correction_sweep_path = (
        # Colab의 임시 저장소 안에만 존재하는 미추적 실행용 YAML 경로를 만듭니다.
        Path(PROJECT_DIR) / "configs/correction/aias_runtime_correction_sweep.yaml"
    )
    # 원본을 건드리지 않고 실행용 보정 YAML을 UTF-8로 저장합니다.
    runtime_correction_sweep_path.write_text(
        # 한글 설정과 키 순서를 유지한 YAML 텍스트를 생성합니다.
        yaml.safe_dump(runtime_correction_sweep, allow_unicode=True, sort_keys=False),
        # 한글이 손상되지 않도록 실행용 YAML을 UTF-8로 저장합니다.
        encoding="utf-8",
    )
    # 이후 보정 탐색 CLI가 세션 ID가 반영된 실행용 명세를 사용하도록 교체합니다.
    CORRECTION_SWEEP_SPEC = str(runtime_correction_sweep_path)

# 이후 AIAS 명령이 기능 토글 반영 설정 사본을 사용하도록 경로를 교체합니다.
CONFIG = str(runtime_config_path)
# 이후 모델 비교가 실행용 설정을 연결한 모델 행렬을 사용하도록 교체합니다.
MODEL_MATRIX = str(runtime_matrix_path)
# 이후 양자화 단계가 활성화 여부를 반영한 임시 명세를 사용하도록 교체합니다.
QUANTIZATION_SPEC = str(runtime_quantization_path)
# 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
print(
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    {
        # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
        "lora": RUN_LORA,
        # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
        "quantization": RUN_QUANTIZATION,
        # Validation 보정 자동 탐색과 성능 게이트 실행 여부를 표시합니다.
        "correction_sweep": RUN_CORRECTION_SWEEP,
        # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
        "information_retrieval": ENABLE_INFORMATION_RETRIEVAL,
        # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
        "nearest_neighbor": ENABLE_NEAREST_NEIGHBOR,
        # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
        "distillation": RUN_DISTILLATION,
        # 새 결과 생성과 기존 설계 결과 재사용 중 현재 선택된 정책을 표시합니다.
        "force_new_experiment": FORCE_NEW_EXPERIMENT,
        # 모든 실험 단계에 공유되는 UTC 세션 ID 또는 재사용 상태를 표시합니다.
        "experiment_session_id": EXPERIMENT_SESSION_ID or "reuse-design-id",
        # 모델 비교 산출물이 저장되는 최종 실험 그룹 ID를 표시합니다.
        "benchmark_id": BENCHMARK_ID,
        # 양자화 산출물이 저장되는 최종 실험 그룹 ID를 표시합니다.
        "quantization_id": QUANTIZATION_ID,
    }
)

## 4. 데이터 준비

`PUBLIC_PROXY`에서는 고정 revision의 `kresnik/zeroth_korean` 일부만 스트리밍해 Drive에 저장합니다. `SYNTHETIC_MANUFACTURING`에서는 저장소에 포함된 TTS WAV와 정답 manifest를 검증합니다. `PRIVATE_MANUFACTURING`에서는 기존 파일을 덮어쓰지 않고 입력 양식을 준비합니다.

In [ ]:
# pathlib 모듈에서 필요한 기능을 불러옵니다.
from pathlib import Path
# json 모듈을 불러옵니다.
import json
# shutil 모듈을 불러옵니다.
import shutil
# pandas 모듈을 불러옵니다.
import pandas as pd

# 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
if IS_PUBLIC_PROXY:
    # 공개 Zeroth 일부를 설정에 맞게 내려받고 manifest·출처 정보를 생성합니다.
    run_aias("prepare-hf-dataset", "--config", CONFIG)
    # 다운로드한 Zeroth 음성과 manifest를 보존할 Drive 폴더를 지정합니다.
    public_root = Path(DRIVE_ROOT) / "data/public/zeroth_korean"
    # 결과를 Colab 표 형태로 표시합니다.
    display(pd.read_csv(public_root / "manifest.csv").groupby("split").size())
    # 현재 데이터셋의 출처·생성 방식·라이선스 정보를 담은 JSON 경로를 지정합니다.
    provenance_path = public_root / "dataset_provenance.json"
    # 데이터 출처·라이선스 JSON이 생성됐는지 확인합니다.
    if provenance_path.exists():
        # 데이터 출처·생성기·개인정보 여부·용도 제한 정보를 화면에 표시합니다.
        display(json.loads(provenance_path.read_text(encoding="utf-8")))
    # 공개 음성 결과를 제조 현장 성능으로 오해하지 않도록 사용 한계를 출력합니다.
    print("PUBLIC PROXY: 제조 성능 증거가 아닌 전체동작 검증 데이터입니다.")
# 합성 제조 TTS 모드일 때 manifest와 출처를 검증합니다.
elif IS_SYNTHETIC_MANUFACTURING:
    # aias_specialist.config 모듈에서 필요한 기능을 불러옵니다.
    from aias_specialist.config import load_settings
    # aias_specialist.data 모듈에서 필요한 기능을 불러옵니다.
    from aias_specialist.data import validate_manifest

    # 합성 제조 manifest와 실험 경로를 YAML 설정에서 읽습니다.
    synthetic_settings = load_settings(CONFIG)
    # 합성 WAV 존재 여부·정답·split·해시를 검증한 manifest를 저장합니다.
    synthetic_manifest = validate_manifest(
        # 설정의 합성 manifest를 Faster-Whisper 입력 규칙으로 검증하도록 전달합니다.
        synthetic_settings.paths.manifest, backend="faster_whisper"
    )
    # train·validation·test와 소음 조건별 합성 음성 개수를 표로 확인합니다.
    display(synthetic_manifest.groupby(["split", "noise_condition"]).size())
    # 현재 데이터셋의 출처·생성 방식·라이선스 정보를 담은 JSON 경로를 지정합니다.
    provenance_path = synthetic_settings.paths.manifest.parent / "dataset_provenance.json"
    # 데이터 출처·생성기·개인정보 여부·용도 제한 정보를 화면에 표시합니다.
    display(json.loads(provenance_path.read_text(encoding="utf-8")))
    # 합성 TTS 결과가 실제 제조·사람 검수 증거가 아님을 출력합니다.
    print("SYNTHETIC: 실제 제조 성능이나 사람 검수 증거가 아닙니다.")
# 공개·합성 모드가 아니므로 실제 제조 데이터 입력 양식을 준비합니다.
else:
    # 실제 제조 데이터의 비공개 Drive 문자열 경로를 Path 객체로 변환합니다.
    private_root = Path(PRIVATE_ROOT)
    # 실제 녹음 WAV를 넣을 비공개 Drive 폴더를 상위 폴더와 함께 생성합니다.
    (private_root / "audio").mkdir(parents=True, exist_ok=True)
    # 실제 제조 모드에서 필요한 manifest·승인·검토·합격기준 양식을 연결합니다.
    templates = {
        # 음성 경로·정답·split·출처를 작성할 manifest CSV 양식을 비공개 폴더에 배치합니다.
        Path("data/templates/manufacturing_manifest_template.csv"): private_root / "manifest.csv",
        # 데이터 사용 승인·동의·비식별화 상태를 기록할 YAML 양식을 배치합니다.
        Path("data/templates/data_approval_template.yaml"): private_root / "data_approval.yaml",
        # 모델·용어·결과의 사람 검토 서명 양식 원본과 비공개 대상 폴더를 연결합니다.
        Path("data/templates/human_review_signoff_template.yaml"): private_root
        # 사람 검토 완료 여부를 기록할 대상 파일명을 human_review_signoff.yaml로 지정합니다.
        / "human_review_signoff.yaml",
        # 정확도·속도·메모리 합격기준 양식 원본과 비공개 대상 폴더를 연결합니다.
        Path("configs/assessment/acceptance_criteria_template.yaml"): private_root
        # 과제 합격기준을 작성할 대상 파일명을 acceptance_criteria.yaml로 지정합니다.
        / "acceptance_criteria.yaml",
    }
    # 각 항목을 순회하며 같은 처리를 반복합니다.
    for source, destination in templates.items():
        # 기존 비공개 입력·검토 문서를 덮어쓰지 않도록 확인합니다.
        if not destination.exists():
            # 원본 양식의 메타데이터를 보존해 비공개 Drive 대상 경로로 복사합니다.
            shutil.copy2(source, destination)
            # 이번 실행에서 새로 만든 입력 양식 경로를 출력합니다.
            print("Created:", destination)
        # 대상 양식이 이미 있으므로 사용자가 작성한 기존 파일을 그대로 보존합니다.
        else:
            # 기존 작성 내용을 덮어쓰지 않고 유지한 파일 경로를 출력합니다.
            print("Preserved existing:", destination)
    # 문서·데이터·실험·거버넌스·사람 검토 증거를 종합 점검합니다.
    run_aias(
        # 심사 증거의 완성도와 차단 항목을 검사하는 감사 명령입니다.
        "assessment-audit",
        # 데이터 경로·split·학습·평가 조건 YAML을 지정하는 옵션입니다.
        "--config",
        # 현재 데이터 모드에 대응하는 핵심 설정 YAML 경로를 전달합니다.
        CONFIG,
        # 감사 결과 JSON과 Markdown을 저장할 폴더 옵션입니다.
        "--output-dir",
        # 실제 제조 입력 준비 상태 점검표를 저장할 Drive 경로를 전달합니다.
        f"{DRIVE_ROOT}/reports/assessment_readiness/private_manufacturing",
    )

## 5. Whisper 모델 비교

공개·합성 모드는 빠른 전체동작 검증을 위해 `tiny`, `base`, `small`을 비교합니다. 실제 제조 모드는 6개 후보를 비교합니다. 모델과 변환본은 Drive 캐시에 재사용됩니다.

In [ ]:
# 모든 Whisper 후보의 Hugging Face revision을 해시로 고정해 재현성을 확보합니다.
run_aias("model-matrix-lock", "--matrix", MODEL_MATRIX)
# 고정된 후보들을 같은 validation split에서 정확도·속도·메모리로 비교합니다.
run_aias("benchmark-models", "--matrix", MODEL_MATRIX)
# 모델별 예측·성능표·보고서가 저장된 Drive 벤치마크 폴더를 지정합니다.
benchmark_dir = Path(DRIVE_ROOT) / "artifacts/benchmarks" / BENCHMARK_ID
# Whisper 후보별 정확도·속도·메모리·용량 비교표를 읽습니다.
benchmark_table = pd.read_csv(benchmark_dir / "benchmark_comparison.csv")
# Whisper 후보별 CER·WER·RTF·메모리·용량·종합순위를 표로 표시합니다.
display(benchmark_table)

## 6. 모델 선택

공개·합성 모드는 제조 용어 Recall 85% 이상을 최우선으로 하고 CER 7% 이하, WER 15% 이하의 목표 부족분을 반영한 rank 1을 자동 선택하지만 사람 검토로 기록하지 않습니다. 실제 제조 모드에서는 아래 사람 검토 값을 직접 입력해야 다음 단계로 진행됩니다.

In [ ]:
# best_completed_member 재사용 함수를 정의합니다.
def best_completed_member(frame):
    # 실패 후보를 제외하고 평가가 완료된 모델 또는 양자화 후보만 남깁니다.
    completed = frame.loc[frame["status"].eq("completed")].copy()
    # 정상 완료된 비교 후보가 하나도 없는지 확인합니다.
    if completed.empty:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise RuntimeError("완료된 후보가 없습니다. 위 오류를 먼저 확인하세요.")
    # 문자열이나 결측값이 섞인 rank 열을 안전한 숫자형으로 변환합니다.
    completed["rank"] = pd.to_numeric(completed["rank"], errors="coerce")
    # 계산하거나 선택한 결과를 호출한 곳에 반환합니다.
    return completed.sort_values(
        # 제조용어 Recall 최우선 목표 게이트, CER, WER, 실시간 처리속도 순위를 따릅니다.
        ["rank", "cer", "wer", "aggregate_real_time_factor"],
        # 순위 정렬 시 결측 성능값을 가장 뒤로 보내도록 지정합니다.
        na_position="last",
    # 정렬 결과의 첫 행, 즉 최우선 완료 후보 하나를 반환합니다.
    ).iloc[0]


# 공개·합성 기능 검증 모드이면 순위 기반 자동 선택을 사용합니다.
if IS_AUTOMATED_PROXY:
    # 완료된 Whisper 후보 중 종합 순위가 가장 높은 행을 선택합니다.
    selected_row = best_completed_member(benchmark_table)
    # 자동 선택된 Whisper 후보의 member_id를 후속 학습 모델로 사용합니다.
    SELECTED_MODEL = str(selected_row["member_id"])
    # 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
    if IS_PUBLIC_PROXY:
        # 공개 프록시의 자동 선택임을 기록해 사람 검토와 구분합니다.
        REVIEWER = "AUTOMATED_PUBLIC_PROXY"
        # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
        MODEL_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "공개 Zeroth 프록시 rank 1 자동 선택. 코드·산출물 smoke test "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "전용이며 제조 모델 선정 또는 사람 검토 증거가 아님."
        )
    # 실제 제조 모드이므로 자동 순위 대신 사람이 모델을 검토하고 입력합니다.
    else:
        # 합성 데이터의 자동 선택임을 기록해 실제 제조 검토와 구분합니다.
        REVIEWER = "AUTOMATED_SYNTHETIC_FIXTURE"
        # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
        MODEL_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "합성 제조 TTS rank 1 자동 선택. 기능 검증 전용이며 실제 제조 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "모델 선정 또는 사람 검토 증거가 아님."
        )
    # 자동 선택 결과를 사람 검토로 오인하지 않도록 프록시 표시 인자를 추가합니다.
    extra_selection_args = ["--automated-proxy"]
# 실제 제조 모드이므로 자동 순위 대신 사람이 모델을 검토하고 입력합니다.
else:
    # 실제 제조 모드에서 비교표 검토 후 선택할 모델의 초기 입력값입니다.
    SELECTED_MODEL = "small"  # 비교표를 보고 수정
    # 실제 제조 모델을 검토한 담당자 이름을 반드시 입력해야 하는 자리입니다.
    REVIEWER = "TO_BE_COMPLETED"
    # 모델 선택의 데이터 범위·정확도·속도·검토 한계를 근거로 기록합니다.
    MODEL_REASON = "TO_BE_COMPLETED: 정확도·속도·메모리·거버넌스 근거"
    # 실제 제조 모델의 검토자 또는 선택 근거가 미입력 상태인지 확인합니다.
    if "TO_BE_COMPLETED" in REVIEWER or "TO_BE_COMPLETED" in MODEL_REASON:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise ValueError("제조 모드에서는 사람 검토자와 모델 선택 근거를 입력하세요.")
    # 실제 제조 모드에서는 자동 프록시 표시 없이 사람 검토 기록을 사용합니다.
    extra_selection_args = []

# 검토한 모델과 선택자·근거를 변경 불가능한 선택 기록으로 남깁니다.
run_aias(
    # 검토한 Whisper 모델을 공식 선택 기록으로 저장하는 명령입니다.
    "select-model",
    # 후보 비교표와 실행 근거가 있는 벤치마크 폴더 옵션입니다.
    "--benchmark-dir",
    # 현재 모드의 벤치마크 산출물 폴더 경로를 전달합니다.
    str(benchmark_dir),
    # 최종 선택할 Whisper 후보 ID를 지정하는 옵션입니다.
    "--model-id",
    # 자동 순위 또는 사람 검토로 선택한 모델 ID를 전달합니다.
    SELECTED_MODEL,
    # 선택을 수행한 자동 절차 또는 사람 검토자를 기록하는 옵션입니다.
    "--reviewer",
    # 현재 선택의 검토자 식별값을 감사 기록에 전달합니다.
    REVIEWER,
    # 정확도·속도·메모리와 데이터 한계를 선택 근거로 기록하는 옵션입니다.
    "--reason",
    # Whisper 모델을 선택한 구체적 판단 근거를 전달합니다.
    MODEL_REASON,
    # 공개·합성 자동 결과인 경우 사람 검토가 아니라는 표시 인자를 펼쳐 전달합니다.
    *extra_selection_args,
)
# 선택 모델·revision·성능·선택 사유가 기록된 YAML 경로를 지정합니다.
model_selection = benchmark_dir / "model_selection.yaml"
# 자동 또는 사람 검토로 선택된 Whisper 모델 ID를 출력합니다.
print("Selected model:", SELECTED_MODEL)
# 저장된 모델 ID·revision·성능·검토자·선택 사유를 화면에서 확인합니다.
print(model_selection.read_text(encoding="utf-8"))

## 6-1. Validation 보정 자동 탐색

선택 모델의 Validation 원본 전사를 재사용해 보정 없음·alias·IR·NN·엄격한 hybrid를 비교합니다. CER·WER·제조용어 Recall·악화 샘플 기준을 모두 통과한 설정만 선택하며, 합격 후보가 없으면 원본 Whisper로 자동 복귀합니다. Test 데이터는 임계값 탐색에 사용하지 않습니다.

In [ ]:
# 보정 탐색을 건너뛴 모드에서도 이후 셀이 안전하게 참조하도록 선택 경로를 초기화합니다.
correction_selection = None
# 현재 모드에 보정 sweep 명세가 있고 실행 옵션이 켜졌는지 확인합니다.
if RUN_CORRECTION_SWEEP and CORRECTION_SWEEP_SPEC:
    # 보정 탐색 명세에서 결과 폴더를 식별할 고유 ID를 읽습니다.
    correction_sweep_spec = yaml.safe_load(
        # 현재 데이터 모드에 연결된 보정 후보·임계값·합격 기준 YAML을 UTF-8로 읽습니다.
        Path(CORRECTION_SWEEP_SPEC).read_text(encoding="utf-8")
    )
    # 명세의 고유 ID를 꺼내 동일한 Drive 결과 폴더를 이후 단계에서도 참조합니다.
    correction_sweep_id = correction_sweep_spec["correction_sweep"]["id"]
    # 선택 모델의 Validation 원본 예측으로 보정 후보와 안전성 기준을 비교합니다.
    run_aias(
        # 원본 전사에 여러 보정 정책을 적용해 비교표를 만드는 CLI 하위 명령입니다.
        "correction-sweep",
        # 보정 후보와 합격 기준 파일을 받는 CLI 옵션입니다.
        "--spec",
        # 현재 실행 모드에 맞는 보정 탐색 YAML 경로를 전달합니다.
        CORRECTION_SWEEP_SPEC,
        # 앞 단계에서 고른 Whisper 모델 선택 기록을 받는 CLI 옵션입니다.
        "--selection",
        # 모델 ID·revision·Validation run 경로가 담긴 선택 YAML을 문자열 경로로 전달합니다.
        str(model_selection),
    )
    # 후보별 CER·WER 개선량과 악화 비율이 저장된 Drive 폴더를 지정합니다.
    correction_sweep_dir = (
        # 세션이 종료되어도 남도록 Drive의 correction 영역과 sweep ID를 결합합니다.
        Path(DRIVE_ROOT) / "artifacts/correction" / correction_sweep_id
    )
    # 후보별 지표·게이트 통과 여부·순위를 Pandas 표로 읽습니다.
    correction_table = pd.read_csv(
        # correction-sweep 명령이 생성한 비교 CSV 파일을 입력으로 지정합니다.
        correction_sweep_dir / "correction_sweep_comparison.csv"
    )
    # 합격 여부·점수 개선·악화율·자동 순위를 표로 표시합니다.
    display(correction_table)
    # 공개·합성 데이터에서는 합격 후보 중 자동 순위 1위를 기능 검증용으로 기록합니다.
    if IS_AUTOMATED_PROXY:
        # 공개 proxy와 합성 fixture 중 어떤 자동 검증 주체인지 구분해 기록합니다.
        correction_reviewer = (
            # 공개 데이터이면 PUBLIC, 합성 데이터이면 SYNTHETIC 검토자 표식을 선택합니다.
            "AUTOMATED_PUBLIC_PROXY" if IS_PUBLIC_PROXY else "AUTOMATED_SYNTHETIC_FIXTURE"
        )
        # 자동 선택이 허용된 범위와 게이트·fallback 근거를 선택 기록에 남깁니다.
        correction_reason = (
            # 정확도와 제조 용어 지표를 동시에 만족해야 한다는 첫 번째 선택 근거입니다.
            "Validation CER·WER·제조용어 Recall·악화 샘플 게이트를 통과한 rank 1 "
            # 합격 후보가 없을 때 원본 Whisper를 유지한다는 안전 동작을 명시합니다.
            "자동 선택. 합격 후보가 없으면 no-correction 안전 기준으로 복귀. "
            # 공개·합성 결과가 실제 제조 성능 승인으로 오해되지 않도록 범위를 제한합니다.
            "기능 검증 전용이며 사람 승인 또는 제조 성능 증거가 아님."
        )
        # CLI에 사람 승인 전 자동 proxy 선택임을 명시하는 옵션을 전달합니다.
        correction_extra_args = ["--automated-proxy"]
    # 비공개 실제 제조 데이터이면 자동 선택하지 않고 사람 검토 절차로 분기합니다.
    else:
        # 실제 제조 데이터에서는 담당자와 선택 근거를 입력한 뒤에만 선택을 기록합니다.
        correction_reviewer = "TO_BE_COMPLETED"
        # Validation 개선·악화 사례를 검토한 구체적인 선택 사유를 사용자가 입력해야 합니다.
        correction_reason = "TO_BE_COMPLETED: Validation 개선·악화 사례 검토 근거"
        # 두 필수 검토 항목 중 하나라도 미입력 상태인지 확인합니다.
        if "TO_BE_COMPLETED" in correction_reviewer or "TO_BE_COMPLETED" in correction_reason:
            # 실제 데이터에서 검토 기록 없이 다음 평가로 넘어가지 못하도록 중단합니다.
            raise ValueError("제조 모드에서는 보정 검토자와 선택 근거를 입력하세요.")
        # 사람 검토 모드에는 automated-proxy 플래그를 전달하지 않습니다.
        correction_extra_args = []
    # 합격 후보 중 rank 1 또는 안전한 no-correction fallback을 선택 기록으로 저장합니다.
    run_aias(
        # sweep 비교표에서 최종 보정 정책을 고정하는 CLI 하위 명령입니다.
        "select-correction",
        # 보정 탐색 결과 폴더를 받는 CLI 옵션입니다.
        "--sweep-dir",
        # 비교 CSV와 후보 설정이 들어 있는 Drive 폴더를 문자열 경로로 전달합니다.
        str(correction_sweep_dir),
        # 선택 기록에 검토 주체를 저장하는 CLI 옵션입니다.
        "--reviewer",
        # 자동 proxy 표식 또는 실제 담당자 이름을 전달합니다.
        correction_reviewer,
        # 선택 이유와 안전성 검토 근거를 저장하는 CLI 옵션입니다.
        "--reason",
        # 위에서 작성한 validation 기반 선택 사유를 전달합니다.
        correction_reason,
        # 공개·합성 모드일 때만 자동 proxy임을 나타내는 추가 인자를 펼쳐 전달합니다.
        *correction_extra_args,
    )
    # CLI가 생성한 최종 보정 정책 선택 YAML의 Drive 경로를 지정합니다.
    correction_selection = correction_sweep_dir / "correction_selection.yaml"
    # 선택 YAML에서 이후 평가에 적용할 correction 설정 블록을 읽습니다.
    selected_correction = yaml.safe_load(
        # 선택 기록 파일을 UTF-8로 읽어 YAML 파서에 전달합니다.
        correction_selection.read_text(encoding="utf-8")
    # selection 메타데이터 아래의 실제 correction 설정만 추출합니다.
    )["selection"]["correction"]
    # 선택된 보정 설정을 LoRA·양자화·최종 Test가 사용하는 실행용 설정에 반영합니다.
    runtime_config["correction"] = selected_correction
    # 갱신한 실행 설정을 Colab 로컬 YAML에 다시 저장합니다.
    runtime_config_path.write_text(
        # 한글과 키 순서를 보존해 수정된 설정 사전을 YAML 문자열로 변환합니다.
        yaml.safe_dump(runtime_config, allow_unicode=True, sort_keys=False),
        # 한글 설정이 손상되지 않도록 UTF-8 인코딩으로 기록합니다.
        encoding="utf-8",
    )
    # 적용된 후보·지표·검토 근거를 사용자가 즉시 확인하도록 선택 YAML을 출력합니다.
    print(correction_selection.read_text(encoding="utf-8"))
# 보정 탐색 옵션이 꺼졌거나 현재 모드에 sweep 명세가 없으면 직접 설정을 사용합니다.
else:
    # 탐색을 끈 경우 앞에서 지정한 직접 보정 토글 설정을 그대로 사용합니다.
    print("Correction sweep skipped; using direct correction switches.")

## 7. 선택 모델 LoRA

공개·합성 모드는 작은 train split과 30 step의 짧은 실행으로 학습 코드, checkpoint, Base/LoRA 비교 산출물을 검증합니다. 선택 모델이 large-v3여도 base weight를 float16·low-memory 방식으로 읽고 micro-batch 1을 사용합니다. 그래도 런타임 자원 한도로 실패하면 실패 증거를 남기고 양자화·최종 Test는 계속합니다. 실제 제조 모드는 500 step을 사용하며 데이터 규모와 GPU에 맞춰 조정합니다.

In [ ]:
# 선택 학습 실패 정보를 후속 산출물 표에서 확인할 수 있도록 초기화합니다.
selected_training_failure = None
# 선택 학습이 실패할 경우 생성할 시간 고정 증거 파일 경로를 미리 비워 둡니다.
selected_training_failure_path = None
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_LORA:
    # 선택 모델 학습만 선택 연구 분기로 격리해 자원 실패가 배포 평가를 막지 않게 합니다.
    try:
        # 선택된 Whisper에 LoRA를 학습하고 Base 대비 결과를 생성합니다.
        run_aias(
            # 선택된 Whisper에 LoRA를 적용하는 학습 명령입니다.
            "train-selected-whisper",
            # 앞 단계의 선택 YAML을 입력으로 받는 옵션입니다.
            "--selection",
            # 모델·revision·실행 조건이 고정된 model_selection.yaml을 전달합니다.
            str(model_selection),
            # 데이터 경로·split·학습·평가 조건 YAML을 지정하는 옵션입니다.
            "--config",
            # 현재 데이터 모드에 대응하는 핵심 설정 YAML 경로를 전달합니다.
            CONFIG,
        )
    # 자식 학습 프로세스가 OOM·SIGKILL·런타임 오류로 종료된 경우를 기록합니다.
    except subprocess.CalledProcessError as exc:
        # 필수 학습 모드에서는 원래 예외를 다시 발생시켜 즉시 중단합니다.
        if not LORA_FAILURE_IS_NON_BLOCKING:
            # 실패 허용이 꺼진 경우 원래 반환 코드와 명령을 보존한 채 호출자에게 전달합니다.
            raise
        # 선택 연구 분기 실패를 숨기지 않고 상태·명령·반환 코드를 구조화합니다.
        selected_training_failure = {
            # 학습 실패가 전체 파이프라인 차단이 아닌 선택 단계 실패임을 표시합니다.
            "status": "failed_optional",
            # 운영체제가 반환한 종료 코드 또는 SIGKILL 신호 값을 보존합니다.
            "returncode": exc.returncode,
            # 재현을 위해 실제 실행된 Python·AIAS CLI 명령 배열을 문자열로 기록합니다.
            "command": [str(value) for value in exc.cmd],
            # 실패 후에도 양자화와 고정 Test를 계속한다는 처리 정책을 기록합니다.
            "message": "LoRA failed; quantization and final Test continue.",
        }
        # 같은 설계의 재실행 기록을 덮어쓰지 않도록 실패 시각 기반 파일명을 만듭니다.
        selected_training_failure_path = (
            # 벤치마크 폴더 아래 별도 실패 이력 폴더와 UTC 시각 파일명을 결합합니다.
            benchmark_dir
            # 선택 학습 실패 증거만 모으는 하위 폴더를 지정합니다.
            / "training_failures"
            # 초 이하 정밀도의 UTC 시각으로 재실행마다 고유한 JSON 파일명을 만듭니다.
            / f"selected_training_failure-{datetime.now(UTC).strftime('%Y%m%dT%H%M%S%fZ')}.json"
        )
        # 아직 없는 실패 이력 폴더를 부모 경로까지 안전하게 생성합니다.
        selected_training_failure_path.parent.mkdir(parents=True, exist_ok=True)
        # 한글을 보존한 JSON으로 선택 학습 실패 증거를 저장합니다.
        selected_training_failure_path.write_text(
            # 구조화한 실패 사전을 사람이 읽기 쉬운 들여쓰기 JSON 문자열로 변환합니다.
            json.dumps(selected_training_failure, ensure_ascii=False, indent=2),
            # 운영체제와 Colab에서 한글 메시지가 깨지지 않도록 UTF-8로 저장합니다.
            encoding="utf-8",
        )
        # 선택 학습 실패와 후속 단계 계속 여부를 실행 로그에 명확히 출력합니다.
        print("Optional LoRA failed; continuing:", selected_training_failure_path)
# 앞 조건이 거짓이므로 이 코드 블록에 정의된 대체 절차를 실행합니다.
else:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("LoRA skipped by RUN_LORA=False")

## 8. Knowledge Distillation (선택)

고정된 Teacher Whisper의 token 분포와 정답 label을 결합해 더 작은 Student를 학습합니다. GPU 시간과 모델 다운로드 비용이 크므로 `RUN_DISTILLATION=True`인 경우에만 실행합니다.

In [ ]:
# 지식 증류를 실행하지 않은 상태를 나타내도록 결과 폴더를 비워 둡니다.
distillation_run_dir = None
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_DISTILLATION:
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    run_aias("model-lock", "--config", CONFIG)
    # 새 증류 실행을 식별하기 위해 실행 전 distill run 폴더 목록을 수집합니다.
    before_distillation = set(Path(DRIVE_ROOT, "artifacts/runs").glob("distill-*"))
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    run_aias("train-whisper-distillation", "--config", CONFIG)
    # 증류 명령 완료 후 distill run 폴더 목록을 다시 수집합니다.
    after_distillation = set(Path(DRIVE_ROOT, "artifacts/runs").glob("distill-*"))
    # 실행 전후 차이로 새 증류 결과만 찾아 수정 시각 순으로 정렬합니다.
    new_distillation_runs = sorted(
        # 이 줄의 값을 바로 위에서 설명한 연구 단계의 입력으로 사용합니다.
        after_distillation - before_distillation,
        # 새 증류 결과 폴더를 파일 수정 시각 기준으로 정렬하는 함수를 전달합니다.
        key=lambda path: path.stat().st_mtime,
    )
    # 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
    if not new_distillation_runs:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise RuntimeError("새 지식 증류 run 디렉터리를 찾지 못했습니다.")
    # 가장 최근에 생성된 지식 증류 run을 결과 확인 대상으로 선택합니다.
    distillation_run_dir = new_distillation_runs[-1]
    # 결과를 Colab 표 형태로 표시합니다.
    display(json.loads((distillation_run_dir / "metrics.json").read_text(encoding="utf-8")))
# 앞 조건이 거짓이므로 이 코드 블록에 정의된 대체 절차를 실행합니다.
else:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("Knowledge distillation skipped by RUN_DISTILLATION=False")

## 9. 양자화 비교

In [ ]:
# 양자화별 예측·성능표·보고서가 저장된 Drive 폴더를 지정합니다.
quantization_dir = Path(DRIVE_ROOT) / "artifacts/quantization" / QUANTIZATION_ID
# 양자화를 끈 경우에도 후속 조건 분기가 가능하도록 비교표를 비워 둡니다.
quantization_table = None
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_QUANTIZATION:
    # 선택 모델의 각 양자화 variant를 동일 조건에서 평가합니다.
    run_aias(
        # 선택 모델의 여러 정밀도를 비교 평가하는 양자화 명령입니다.
        "quantization-sweep",
        # 양자화 후보·캐시·평가 기준 YAML을 지정하는 옵션입니다.
        "--spec",
        # 현재 모드의 float16·int8 비교 명세 경로를 전달합니다.
        QUANTIZATION_SPEC,
        # 앞 단계의 선택 YAML을 입력으로 받는 옵션입니다.
        "--selection",
        # 모델·revision·실행 조건이 고정된 model_selection.yaml을 전달합니다.
        str(model_selection),
    )
    # 양자화 후보별 정확도 변화·속도·메모리·용량 비교표를 읽습니다.
    quantization_table = pd.read_csv(quantization_dir / "quantization_comparison.csv")
    # 정밀도별 정확도 변화·속도·메모리·용량·종합순위를 표로 표시합니다.
    display(quantization_table)
# 앞 조건이 거짓이므로 이 코드 블록에 정의된 대체 절차를 실행합니다.
else:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("Quantization skipped by RUN_QUANTIZATION=False")

## 10. 양자화 선택

공개·합성 모드는 종합 rank 1을 자동 선택합니다. 실제 제조 모드는 정확도 손실, RTF, GPU 메모리와 모델 용량을 사람이 함께 검토합니다.

In [ ]:
# 양자화를 끈 경우 모델 선택을 최종 선택으로 사용할 수 있게 초기화합니다.
quantization_selection = None
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if not RUN_QUANTIZATION:
    # 해당 단계의 상태·선택 근거·안내 문구를 실행 로그에 출력합니다.
    print("Quantization selection skipped because quantization is disabled.")
# 앞 모드가 아닐 때 이 줄의 다음 데이터 모드 조건을 검사합니다.
elif IS_AUTOMATED_PROXY:
    # 완료된 양자화 후보 중 종합 순위가 가장 높은 행을 선택합니다.
    selected_quantization_row = best_completed_member(quantization_table)
    # 자동 선택된 양자화 variant ID를 최종 Test 평가에 사용합니다.
    SELECTED_VARIANT = str(selected_quantization_row["member_id"])
    # 공개 Zeroth 프록시 모드일 때의 데이터·선택 절차를 실행합니다.
    if IS_PUBLIC_PROXY:
        # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
        QUANTIZATION_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "공개 Zeroth 프록시 종합 rank 1 자동 선택. 양자화 코드·산출물 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "smoke test 전용이며 실제 배포 결정 또는 사람 검토 증거가 아님."
        )
    # 실제 제조 모드이므로 사람이 양자화 손실과 자원 절감을 검토합니다.
    else:
        # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
        QUANTIZATION_REASON = (
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "합성 제조 TTS 종합 rank 1 자동 선택. 기능 검증 전용이며 실제 "
            # 자동 프록시 결과의 범위와 사람 검토가 아님을 선택 근거에 명시합니다.
            "배포 결정 또는 사람 검토 증거가 아님."
        )
    # 양자화 자동 선택을 사람 배포 결정으로 오인하지 않도록 프록시 표시를 추가합니다.
    extra_quantization_args = ["--automated-proxy"]
# 앞 모드가 아닐 때 이 줄의 다음 데이터 모드 조건을 검사합니다.
elif RUN_QUANTIZATION:
    # 실제 제조 모드에서 손실·속도·메모리를 검토한 뒤 선택할 초기 양자화 값입니다.
    SELECTED_VARIANT = "float16"  # 비교표를 보고 수정
    # 양자화 선택의 정확도 손실·속도·메모리·용량 근거를 기록합니다.
    QUANTIZATION_REASON = "TO_BE_COMPLETED: 정확도 손실·속도·메모리·모델 용량 근거"
    # 실제 제조 양자화 선택 근거가 미입력 상태인지 확인합니다.
    if "TO_BE_COMPLETED" in QUANTIZATION_REASON:
        # 필수 조건을 만족하지 않으면 명확한 오류로 실행을 중단합니다.
        raise ValueError("제조 모드에서는 양자화 선택 근거를 입력하세요.")
    # 실제 제조 모드에서는 자동 프록시 표시 없이 사람의 양자화 결정을 기록합니다.
    extra_quantization_args = []

# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_QUANTIZATION:
    # 검토한 양자화와 선택자·근거를 선택 기록으로 남깁니다.
    run_aias(
        # 검토한 양자화 variant를 공식 선택 기록으로 저장하는 명령입니다.
        "select-quantization",
        # 양자화 비교표와 실행 근거가 있는 폴더 옵션입니다.
        "--quantization-dir",
        # 현재 모드의 양자화 비교 산출물 폴더를 전달합니다.
        str(quantization_dir),
        # 최종 선택할 정밀도 variant ID를 지정하는 옵션입니다.
        "--variant-id",
        # 자동 순위 또는 사람 검토로 선택한 양자화 ID를 전달합니다.
        SELECTED_VARIANT,
        # 선택을 수행한 자동 절차 또는 사람 검토자를 기록하는 옵션입니다.
        "--reviewer",
        # 현재 선택의 검토자 식별값을 감사 기록에 전달합니다.
        REVIEWER,
        # 정확도·속도·메모리와 데이터 한계를 선택 근거로 기록하는 옵션입니다.
        "--reason",
        # 정확도 손실·속도·메모리·용량을 고려한 선택 근거를 전달합니다.
        QUANTIZATION_REASON,
        # 공개·합성 자동 결과인 경우 배포 결정이 아니라는 표시 인자를 펼쳐 전달합니다.
        *extra_quantization_args,
    )
    # 최종 variant·성능·선택 사유가 기록된 YAML 경로를 지정합니다.
    quantization_selection = quantization_dir / "quantization_selection.yaml"
    # 자동 또는 사람 검토로 선택된 양자화 variant ID를 출력합니다.
    print("Selected variant:", SELECTED_VARIANT)
    # 저장된 variant·성능·검토자·선택 사유를 화면에서 확인합니다.
    print(quantization_selection.read_text(encoding="utf-8"))

## 11. 고정 Test 최종평가

모델·양자화 선택이 끝난 뒤에만 그동안 보지 않은 Test split을 한 번 평가합니다. 실행 후 보정 전후 WER·CER·제조 용어 Recall과 개선·악화 문장 수를 표로 확인합니다.

In [ ]:
# 양자화 실행 여부에 따라 최종 Test에 사용할 선택 YAML을 결정합니다.
final_selection = quantization_selection if RUN_QUANTIZATION else model_selection
# 선택이 끝난 모델을 미사용 고정 Test split에서 한 번 평가합니다.
run_aias(
    # 선택 완료 후 미사용 Test split을 평가하는 최종 명령입니다.
    "finalize-evaluation",
    # 앞 단계의 선택 YAML을 입력으로 받는 옵션입니다.
    "--selection",
    # 이 줄의 값을 바로 위에서 설명한 연구 단계의 입력으로 사용합니다.
    str(final_selection),
    # 데이터 경로·split·학습·평가 조건 YAML을 지정하는 옵션입니다.
    "--config",
    # 현재 데이터 모드에 대응하는 핵심 설정 YAML 경로를 전달합니다.
    CONFIG,
)
# CLI가 저장한 고정 Test 결과 JSON 경로를 선택 파일과 같은 폴더에서 찾습니다.
final_test_result_path = final_selection.parent / "final_test_result.json"
# 실행 run 경로와 보정 전후 전체 지표가 담긴 최종 Test 결과를 읽습니다.
final_test_result = json.loads(
    # 한글 경로와 JSON 내용을 안전하게 처리하도록 UTF-8로 파일을 읽습니다.
    final_test_result_path.read_text(encoding="utf-8")
)
# 최종 Test 결과에서 원본 Whisper와 선택 보정의 평가 지표 묶음을 꺼냅니다.
final_test_metrics = final_test_result["metrics"]
# 후처리 적용 전 Whisper 원본의 WER·CER·용어 Recall을 저장합니다.
final_test_baseline = final_test_metrics["baseline"]
# 선택된 Alias·IR·NN 후처리 적용 후의 동일 Test 지표를 저장합니다.
final_test_corrected = final_test_metrics["corrected"]
# 오류율과 Recall의 개선 방향 차이를 일관된 양수 개선량으로 변환하는 함수를 정의합니다.
def build_test_metric_row(label, metric_key, higher_is_better):
    # 지정 지표의 후처리 전 값을 실수형으로 읽습니다.
    before = float(final_test_baseline[metric_key])
    # 같은 지표의 후처리 후 값을 실수형으로 읽습니다.
    after = float(final_test_corrected[metric_key])
    # Recall은 증가분, 오류율은 감소분이 양수가 되도록 개선량을 계산합니다.
    improvement = after - before if higher_is_better else before - after
    # 표 한 행에 지표명·전후 값·절대 개선량을 함께 반환합니다.
    return {
        # 사람이 지표의 의미와 좋은 방향을 바로 알 수 있는 이름입니다.
        "metric": label,
        # 선택 후처리를 적용하기 전 고정 Test 성능입니다.
        "before": before,
        # 선택 후처리를 적용한 뒤의 고정 Test 성능입니다.
        "after": after,
        # 양수이면 개선, 음수이면 악화를 뜻하는 절대 변화량입니다.
        "absolute_improvement": improvement,
    }
# 고정 Test의 정확도와 제조 용어 인식 변화를 세 행의 비교표로 만듭니다.
final_test_metric_table = pd.DataFrame(
    # WER·CER·용어 Recall에 동일한 전후 비교 계산을 적용합니다.
    [
        # 단어 오류율은 낮을수록 좋으므로 감소량을 개선값으로 표시합니다.
        build_test_metric_row("WER (낮을수록 좋음)", "wer", False),
        # 글자 오류율도 낮을수록 좋으므로 감소량을 개선값으로 표시합니다.
        build_test_metric_row("CER (낮을수록 좋음)", "cer", False),
        # 제조 용어 Recall은 높을수록 좋으므로 증가량을 개선값으로 표시합니다.
        build_test_metric_row(
            # 표에서 제조 용어 재현율의 좋은 방향을 명시합니다.
            "제조 용어 Recall (높을수록 좋음)",
            # metrics.json에서 제조 용어 Recall 값을 찾는 키입니다.
            "domain_term_recall",
            # Recall은 후처리 후 값이 증가할수록 개선임을 지정합니다.
            True,
        ),
    ]
)
# 후처리 전후 성능과 양수 기준 개선량을 Colab 표로 표시합니다.
display(final_test_metric_table)
# 최종 corrected 지표가 현업 목표 세 가지를 모두 만족하는지 판정한 결과를 읽습니다.
final_quality_gate = final_test_result["quality_gate"]
# Recall 최우선, CER·WER 순서로 목표·관측값·부족분·통과 여부를 표로 만듭니다.
final_quality_target_table = pd.DataFrame(
    # 세 현업 지표를 최우선 순서대로 표시할 행 목록을 시작합니다.
    [
        # 제조 용어 Recall의 목표·관측값·부족분·통과 여부를 첫 번째 행에 기록합니다.
        {
            # 현업 최우선 지표임을 순위 1로 표시합니다.
            "priority": 1,
            # 표에 표시할 제조 용어 인식 지표 이름입니다.
            "metric": "제조 용어 Recall",
            # 현업에서 요구한 최소 제조 용어 Recall 기준입니다.
            "target": ">= 85%",
            # 최종 Test 후처리 결과의 실제 제조 용어 Recall입니다.
            "observed": final_quality_gate["observed"]["domain_term_recall"],
            # 목표 85%에 부족한 절대 비율을 기록합니다.
            "gap": final_quality_gate["checks"]["domain_term_recall"]["gap"],
            # 실제 Recall이 목표 이상인지 여부를 표시합니다.
            "passed": final_quality_gate["checks"]["domain_term_recall"]["passed"],
        },
        # CER의 목표·관측값·초과분·통과 여부를 두 번째 행에 기록합니다.
        {
            # 체감 품질 지표임을 순위 2로 표시합니다.
            "priority": 2,
            # 표에 표시할 글자 오류율 지표 이름입니다.
            "metric": "CER",
            # 현업에서 요구한 최대 글자 오류율 기준입니다.
            "target": "<= 7%",
            # 최종 Test 후처리 결과의 실제 CER입니다.
            "observed": final_quality_gate["observed"]["cer"],
            # 목표 7%를 초과한 절대 비율을 기록합니다.
            "gap": final_quality_gate["checks"]["cer"]["gap"],
            # 실제 CER이 목표 이하인지 여부를 표시합니다.
            "passed": final_quality_gate["checks"]["cer"]["passed"],
        },
        # WER의 목표·관측값·초과분·통과 여부를 세 번째 행에 기록합니다.
        {
            # 참고 단어 오류율 지표임을 순위 3으로 표시합니다.
            "priority": 3,
            # 표에 표시할 단어 오류율 지표 이름입니다.
            "metric": "WER",
            # 현업에서 요구한 최대 단어 오류율 기준입니다.
            "target": "<= 15%",
            # 최종 Test 후처리 결과의 실제 WER입니다.
            "observed": final_quality_gate["observed"]["wer"],
            # 목표 15%를 초과한 절대 비율을 기록합니다.
            "gap": final_quality_gate["checks"]["wer"]["gap"],
            # 실제 WER이 목표 이하인지 여부를 표시합니다.
            "passed": final_quality_gate["checks"]["wer"]["passed"],
        },
    ]
)
# 목표 미달을 숨기지 않고 각 지표의 남은 개선 폭을 Colab에 표시합니다.
display(final_quality_target_table)
# 세 목표의 동시 통과 여부를 사람이 바로 읽을 수 있는 상태 문자열로 바꿉니다.
final_quality_target_status = (
    # 세 목표가 모두 통과했으면 PASS로 표시합니다.
    "PASS"
    # 전체 목표 통과 여부에 따라 PASS 또는 개선 필요 상태를 선택합니다.
    if final_quality_gate["overall_pass"]
    # 하나라도 미달이면 현업 배포 전 개선이 필요함을 표시합니다.
    else "FAIL - 모델·데이터·후처리 개선 필요"
)
# 현업 정확도 합격 여부를 Colab 실행 로그에 명확히 출력합니다.
print("현업 정확도 목표:", final_quality_target_status)
# 최종 Test run에서 문장별 보정 내용과 판정이 기록된 감사 CSV 경로를 만듭니다.
final_test_audit_path = Path(final_test_result["run_dir"]) / "correction_audit.csv"
# 문장별 원문·보정문·개선·악화 판정을 담은 감사 데이터를 읽습니다.
final_test_audit = pd.read_csv(final_test_audit_path)
# CSV의 True 문자열 또는 불리언 값을 모두 처리해 실제 변경 문장 마스크를 만듭니다.
final_test_changed_mask = (
    # text_changed 열을 문자열로 통일하고 대소문자 차이를 제거합니다.
    final_test_audit["text_changed"].astype(str).str.lower().eq("true")
)
# improved·degraded·unchanged 판정별 문장 개수를 집계합니다.
final_test_outcome_counts = final_test_audit["outcome"].value_counts()
# 전체·변경·개선·악화·미변경 문장 수를 한 행의 요약표로 만듭니다.
final_test_change_table = pd.DataFrame(
    # 현재 최종 Test 실행 하나를 나타내는 요약 행 목록을 시작합니다.
    [
        # 문장 수 집계 항목을 열로 가지는 하나의 사전을 구성합니다.
        {
            # 고정 Test에 포함된 전체 문장 수입니다.
            "sample_count": len(final_test_audit),
            # 후처리로 인식 문장이 실제 변경된 수입니다.
            "changed_count": int(final_test_changed_mask.sum()),
            # CER 우선, 동률 시 WER 기준으로 개선된 문장 수입니다.
            "improved_count": int(final_test_outcome_counts.get("improved", 0)),
            # 후처리 후 오류가 늘어난 문장 수입니다.
            "degraded_count": int(final_test_outcome_counts.get("degraded", 0)),
            # 후처리 전후 점수가 동일한 문장 수입니다.
            "unchanged_count": int(final_test_outcome_counts.get("unchanged", 0)),
        }
    ]
)
# 후처리 변경이 전체 Test에 미친 개선·악화 문장 수를 표로 표시합니다.
display(final_test_change_table)
# 사용자가 검토할 문장별 전후 비교표의 핵심 열 순서를 정의합니다.
final_test_review_columns = [
    # 각 음성 샘플을 추적할 고유 ID입니다.
    "sample_id",
    # 후처리가 해당 샘플을 개선·악화·유지했는지 나타냅니다.
    "outcome",
    # 평가 정답으로 사용하는 기준 전사 문장입니다.
    "reference_text",
    # 제조 용어 후처리 적용 전 Whisper 인식 문장입니다.
    "recognized_before",
    # 선택된 Alias·IR·NN 후처리를 적용한 문장입니다.
    "corrected_after",
    # 실제 문장 변경에 사용된 보정 방법입니다.
    "correction_methods",
    # 후처리로 감소한 글자 오류율이며 양수이면 개선입니다.
    "cer_absolute_reduction",
]
# 실제로 변경된 문장이 하나 이상 있는지 확인합니다.
if final_test_changed_mask.any():
    # 화면이 과도하게 길어지지 않도록 변경 사례 중 앞 20개만 표시합니다.
    display(
        # 변경된 행과 검토 핵심 열을 선택한 뒤 최대 20개로 제한합니다.
        final_test_audit.loc[final_test_changed_mask, final_test_review_columns].head(20)
    )
# 선택 보정이 Test 문장을 하나도 바꾸지 않은 경우의 안내를 처리합니다.
else:
    # 변경 없음이 정상 계산 결과임을 사용자에게 명확히 알립니다.
    print("고정 Test에서 후처리로 변경된 문장이 없습니다.")
# 전체 문장별 전후 로그가 보존된 Drive 파일 경로를 출력합니다.
print("전체 고정 Test 보정 로그:", final_test_audit_path)

## 12. 산출물·심사 준비도 확인

공개·합성 모드에서는 `not_ready`가 정상입니다. 공개·합성 데이터, 자동 선택, 미완료 사람 서명은 제조 심사 증거를 대체하지 못합니다.

In [ ]:
# 현재 데이터 모드의 심사 준비도 JSON·Markdown을 저장할 Drive 폴더를 지정합니다.
readiness_dir = Path(DRIVE_ROOT) / "reports/assessment_readiness" / DATA_MODE.lower()
# 문서·데이터·실험·거버넌스·사람 검토 증거를 종합 점검합니다.
run_aias(
    # 심사 증거의 완성도와 차단 항목을 검사하는 감사 명령입니다.
    "assessment-audit",
    # 데이터 경로·split·학습·평가 조건 YAML을 지정하는 옵션입니다.
    "--config",
    # 현재 데이터 모드에 대응하는 핵심 설정 YAML 경로를 전달합니다.
    CONFIG,
    # 감사 결과 JSON과 Markdown을 저장할 폴더 옵션입니다.
    "--output-dir",
    # 현재 모드의 심사 준비도 보고서 폴더를 전달합니다.
    str(readiness_dir),
)
# 심사 항목별 통과·대기·실패 상태가 담긴 JSON 결과를 읽습니다.
readiness = json.loads((readiness_dir / "assessment_readiness.json").read_text(encoding="utf-8"))
# 현재 모드의 최종 심사 준비 상태를 출력합니다.
print("Assessment readiness:", readiness["overall_status"])
# 결과를 Colab 표 형태로 표시합니다.
display(
    # 심사기준·검사항목·상태·설명·증거 경로 열만 선택해 검토표를 만듭니다.
    pd.DataFrame(readiness["checks"])[["criterion", "check_id", "status", "message", "evidence"]]
)

## 13. 생성 결과 위치 요약

In [ ]:
# 최종 확인할 필수 표·보고서·선택 기록·백데이터 목록을 정의합니다.
expected_outputs = {
    # Whisper 모델 후보 비교 CSV 경로를 등록합니다.
    "benchmark_table": benchmark_dir / "benchmark_comparison.csv",
    # 모델 비교 결과 Word 보고서 경로를 등록합니다.
    "benchmark_report": benchmark_dir / "reports/benchmark_report.docx",
    # 선택 모델과 근거를 기록한 YAML 경로를 등록합니다.
    "model_selection": benchmark_dir / "model_selection.yaml",
    # 선택 완료 후 고정 Test 결과 JSON 경로를 등록합니다.
    "final_test": final_selection.parent / "final_test_result.json",
    # 심사 준비도 기계 판독용 JSON 경로를 등록합니다.
    "readiness_json": readiness_dir / "assessment_readiness.json",
    # 심사 준비도 사람이 읽을 Markdown 경로를 등록합니다.
    "readiness_markdown": readiness_dir / "assessment_readiness.md",
    # 모든 실험 이력을 누적한 SQLite 백데이터 경로를 등록합니다.
    "experiment_database": Path(DRIVE_ROOT) / "backdata/experiments.sqlite3",
}
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_LORA:
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    expected_outputs["selected_training"] = benchmark_dir / "selected_training_result.json"
    # 이번 실행에서 LoRA가 실패했다면 시간 고정 실패 JSON도 최종 산출물 표에 포함합니다.
    if selected_training_failure_path is not None:
        # 선택 학습 실패 원인과 후속 단계 계속 정책이 담긴 증거 파일을 등록합니다.
        expected_outputs["selected_training_failure"] = selected_training_failure_path
# 보정 자동 탐색을 실행했으면 비교표·보고서·선택 기록을 최종 산출물에 포함합니다.
if correction_selection is not None:
    # 보정 탐색에서 생성된 세 파일을 최종 산출물 확인 사전에 한꺼번에 추가합니다.
    expected_outputs.update(
        # 산출물의 의미를 나타내는 이름과 실제 Drive 경로를 대응시킵니다.
        {
            # 후보별 정확도 변화·악화율·게이트 통과 여부가 담긴 비교 CSV입니다.
            "correction_sweep_table": correction_sweep_dir / "correction_sweep_comparison.csv",
            # 보정 후보 비교와 권고 결과를 정리한 Word 보고서 경로입니다.
            "correction_sweep_report": (
                # sweep 결과 폴더 아래 reports 디렉터리의 보고서 파일을 지정합니다.
                correction_sweep_dir / "reports/correction_sweep_report.docx"
            ),
            # 최종 선택 정책·Validation 지표·검토 근거가 담긴 YAML 경로입니다.
            "correction_selection": correction_selection,
        }
    )
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if RUN_QUANTIZATION:
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    expected_outputs.update(
        # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
        {
            # 양자화 후보 비교 CSV 경로를 등록합니다.
            "quantization_table": quantization_dir / "quantization_comparison.csv",
            # 양자화 비교 결과 Word 보고서 경로를 등록합니다.
            "quantization_report": (quantization_dir / "reports/quantization_report.docx"),
            # 선택 양자화와 근거를 기록한 YAML 경로를 등록합니다.
            "quantization_selection": quantization_selection,
        }
    )
# 이 줄에 명시된 보호 조건을 검사해 다음 처리 경로를 결정합니다.
if distillation_run_dir is not None:
    # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
    expected_outputs.update(
        # 이 연산으로 현재 데이터 준비·평가·선택 단계의 상태를 갱신합니다.
        {
            # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
            "distillation_metrics": distillation_run_dir / "metrics.json",
            # 이 키가 나타내는 세부 설정값을 현재 실행 모드에 연결합니다.
            "distillation_report": (distillation_run_dir / "reports/evaluation_report.docx"),
        }
    )
# 결과를 Colab 표 형태로 표시합니다.
display(
    # 산출물 존재 여부 목록을 Colab에서 읽기 쉬운 표로 변환합니다.
    pd.DataFrame(
        # 필수 산출물마다 존재 여부와 경로를 계산할 행 목록을 시작합니다.
        [
            # 각 산출물의 이름·존재 여부·실제 저장 경로를 한 행으로 구성합니다.
            {"artifact": name, "exists": path.exists(), "path": str(path)}
            # 각 항목을 순회하며 같은 처리를 반복합니다.
            for name, path in expected_outputs.items()
        ]
    )
)

# 공개·합성 기능 검증 모드이면 순위 기반 자동 선택을 사용합니다.
if IS_AUTOMATED_PROXY:
    # 기능 검증 완료 후 실제 제조 데이터 모드로 전환하는 다음 단계를 출력합니다.
    print(
        # 기능 검증 다음에는 실행 모드를 실제 제조 데이터로 바꾸라고 안내합니다.
        "다음 단계: DATA_MODE을 PRIVATE_MANUFACTURING으로 바꾸고 승인된 제조 "
        # 승인된 녹음·정답으로 동일 파이프라인을 재실행하라는 절차를 완성합니다.
        "녹음·정답 전사를 넣은 뒤 같은 순서를 다시 실행합니다."
    )
# 실제 제조 모드의 최종 사람 검수와 서명 완료 절차를 안내합니다.
else:
    # 실제 제조 결과의 사람 검수·서명·최종 감사 절차를 출력합니다.
    print(
        # 실제 제조 결과의 보고서·오류 샘플·사람 검토 서명을 요구합니다.
        "최종 보고서와 오류 샘플을 사람이 검수하고 human_review_signoff.yaml을 "
        # 서명 후 차단 항목이 남으면 실패하는 최종 감사 명령을 안내합니다.
        "완료한 뒤 assessment-audit --fail-on-blocker로 최종 확인하세요."
    )